# 01d — Normalisations & Export

In [ ]:

import os, sys, numpy as np, pandas as pd
sys.path.append(os.path.join("..","src"))
from raman_ai.io.standards import load_reference_spectra_mode

def normalize(y, mode="area", wn=None):
    y = np.asarray(y, float)
    if mode == "max":
        m = np.max(np.abs(y)) + 1e-12
        return y / m
    elif mode == "area":
        if wn is None:
            return y / (np.trapz(y) + 1e-12)
        else:
            return y / (np.trapz(y, wn) + 1e-12)
    elif mode == "zscore":
        mu, sd = np.mean(y), np.std(y) + 1e-12
        return (y - mu) / sd
    else:
        return y

root = os.path.join("..","data","standards")
specs, metas = load_reference_spectra_mode(root, mode="synthetic")
s = specs[0]
wn = np.asarray(s.wavenumber, float)
y = np.asarray(s.intensity, float)
yn = normalize(y, mode="area", wn=wn)

import matplotlib.pyplot as plt
plt.figure(); plt.plot(wn, yn); plt.title("Normalized (area)"); plt.xlabel("cm^-1"); plt.ylabel("a.u."); plt.tight_layout(); plt.show()

outp = os.path.join("..","data","processed"); os.makedirs(outp, exist_ok=True)
pd.DataFrame({"wavenumber": wn, "intensity": yn}).to_parquet(os.path.join(outp, f"{s.name}_preprocessed.parquet"))
